In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amirhosseinmirzaie/americancitizenincome")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'americancitizenincome' dataset.
Path to dataset files: /kaggle/input/americancitizenincome


In [7]:
import os
import pandas as pd
import numpy as np
import kagglehub

# Force check and download the correct path for americancitizenincome
path = kagglehub.dataset_download("amirhosseinmirzaie/americancitizenincome")
print("Correct path to dataset files:", path)

# Find the CSV file inside the downloaded path
files = os.listdir(path)
csv_files = [f for f in files if f.endswith('.csv')]
if not csv_files:
    for root, dirs, filenames in os.walk(path):
        for f in filenames:
            if f.endswith('.csv'):
                csv_files.append(os.path.join(root, f))

selected_file = csv_files[0] if '/' in csv_files[0] else os.path.join(path, csv_files[0])
print("Loading file:", selected_file)

# Use 'latin-1' encoding to resolve UnicodeDecodeError
df = pd.read_csv(selected_file, encoding='latin-1')
print("Dataset shape:", df.shape)
display(df.head())
print(df.info())

Using Colab cache for faster access to the 'americancitizenincome' dataset.
Correct path to dataset files: /kaggle/input/americancitizenincome
Loading file: /kaggle/input/americancitizenincome/income.csv
Dataset shape: (25000, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,40,Self-emp-not-inc,223881,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,99999,0,70,United-States,>50K
1,30,Private,149118,HS-grad,9,Divorced,Craft-repair,Not-in-family,White,Female,0,0,40,United-States,<=50K
2,46,Private,109209,Some-college,10,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,40,United-States,>50K
3,32,Private,229566,Assoc-voc,11,Married-civ-spouse,Other-service,Husband,White,Male,0,0,60,United-States,>50K
4,54,?,148657,Preschool,1,Married-civ-spouse,?,Wife,White,Female,0,0,40,Mexico,<=50K


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             25000 non-null  int64 
 1   workclass       25000 non-null  object
 2   fnlwgt          25000 non-null  int64 
 3   education       25000 non-null  object
 4   education.num   25000 non-null  int64 
 5   marital.status  25000 non-null  object
 6   occupation      25000 non-null  object
 7   relationship    25000 non-null  object
 8   race            25000 non-null  object
 9   sex             25000 non-null  object
 10  capital.gain    25000 non-null  int64 
 11  capital.loss    25000 non-null  int64 
 12  hours.per.week  25000 non-null  int64 
 13  native.country  25000 non-null  object
 14  income          25000 non-null  object
dtypes: int64(6), object(9)
memory usage: 2.9+ MB
None


In [8]:
# Clean and Preprocess the Income Dataset
# We'll drop non-predictive or redundant columns
df_clean = df.copy()
if 'education' in df_clean.columns and 'education-num' in df_clean.columns:
    df_clean = df_clean.drop(columns=['education'])

# Identify categorical columns and use one-hot encoding
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
# If the target label column (typically 'income' or 'class') is categorical, we will handle it separately
target_col = [col for col in ['income', 'class', 'income_class'] if col in df_clean.columns]
if target_col:
    target_col = target_col[0]
    # Convert target binary classes to 0 and 1
    df_clean[target_col] = df_clean[target_col].astype(str).str.contains('>50K').astype(int)
    categorical_cols = [c for c in categorical_cols if c != target_col]
else:
    target_col = df_clean.columns[-1]

df_clean = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

X = df_clean.drop(columns=[target_col]).values.astype(float)
y = df_clean[target_col].values.reshape(-1, 1).astype(float)

# Scale data using Min-Max scaling
X_min, X_max = X.min(axis=0), X.max(axis=0)
X_max[X_max == X_min] = X_max[X_max == X_min] + 1e-8
X_scaled = (X - X_min) / (X_max - X_min)

# Train-Test Split (80/20)
np.random.seed(42)
indices = np.random.permutation(len(X_scaled))
train_size = int(0.8 * len(X_scaled))
train_idx, test_idx = indices[:train_size], indices[train_size:]

X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Inputs shape: {X_train.shape}")
print(f"Targets shape: {y_train.shape}")

Inputs shape: (20000, 99)
Targets shape: (20000, 1)


In [9]:
class BackpropANN:
    def __init__(self, layers, learning_rate=0.01):
        self.layers = layers
        self.lr = learning_rate
        self.weights = []
        self.biases = []

        # Xavier initialization
        for i in range(len(layers) - 1):
            w = np.random.randn(layers[i], layers[i+1]) * np.sqrt(2.0 / (layers[i] + layers[i+1]))
            b = np.zeros((1, layers[i+1]))
            self.weights.append(w)
            self.biases.append(b)

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def sigmoid_derivative(self, a):
        return a * (1 - a)

    def forward(self, X):
        self.activations = [X]
        self.zs = []

        # Feedforward through all layers with sigmoid activation
        for i in range(len(self.weights)):
            z = np.dot(self.activations[-1], self.weights[i]) + self.biases[i]
            self.zs.append(z)
            self.activations.append(self.sigmoid(z))
        return self.activations[-1]

    def backward(self, X, y, output):
        # For binary classification using Sigmoid Output & Binary Cross-Entropy loss,
        # the derivative dLoss/dZ at the output layer simplifies directly to (output - y)
        error = output - y
        deltas = [error]

        # Propagate error backwards through the hidden layers
        for i in range(len(self.weights) - 1, 0, -1):
            delta = np.dot(deltas[-1], self.weights[i].T) * self.sigmoid_derivative(self.activations[i])
            deltas.append(delta)

        deltas.reverse()

        # Update weights and biases using mini-batch gradient updates
        m = X.shape[0]
        for i in range(len(self.weights)):
            dw = np.dot(self.activations[i].T, deltas[i]) / m
            db = np.sum(deltas[i], axis=0, keepdims=True) / m
            self.weights[i] -= self.lr * dw
            self.biases[i] -= self.lr * db

    def train(self, X, y, epochs=100, batch_size=64):
        loss_history = []
        for epoch in range(epochs):
            indices = np.random.permutation(X.shape[0])
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for i in range(0, X.shape[0], batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                output = self.forward(X_batch)
                self.backward(X_batch, y_batch, output)

            # Track Binary Cross Entropy Loss
            full_output = self.forward(X)
            epsilon = 1e-15
            full_output = np.clip(full_output, epsilon, 1 - epsilon)
            epoch_loss = -np.mean(y * np.log(full_output) + (1 - y) * np.log(1 - full_output))
            loss_history.append(epoch_loss)

            if (epoch + 1) % (epochs // 10 or 1) == 0 or epoch == 0:
                print(f"Epoch {epoch+1:3d}/{epochs} - Loss (Binary Cross-Entropy): {epoch_loss:.6f}")
        return loss_history

In [10]:
import matplotlib.pyplot as plt

# Configure & train the classification ANN model
input_dim = X_train.shape[1]
network = BackpropANN(layers=[input_dim, 32, 16, 1], learning_rate=0.1)

print("Starting training on American Citizen Income dataset...")
history = network.train(X_train, y_train, epochs=100, batch_size=64)

# Evaluate predictions on the test set
predictions_prob = network.forward(X_test)
predictions_class = (predictions_prob >= 0.5).astype(int)

# Metrics calculation
accuracy = np.mean(predictions_class == y_test)
print(f"\nEvaluation Metrics on Test Set:")
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Starting training on American Citizen Income dataset...
Epoch   1/100 - Loss (Binary Cross-Entropy): 0.546278
Epoch  10/100 - Loss (Binary Cross-Entropy): 0.359612
Epoch  20/100 - Loss (Binary Cross-Entropy): 0.351178
Epoch  30/100 - Loss (Binary Cross-Entropy): 0.335958
Epoch  40/100 - Loss (Binary Cross-Entropy): 0.333346
Epoch  50/100 - Loss (Binary Cross-Entropy): 0.329674
Epoch  60/100 - Loss (Binary Cross-Entropy): 0.327044
Epoch  70/100 - Loss (Binary Cross-Entropy): 0.339279
Epoch  80/100 - Loss (Binary Cross-Entropy): 0.327017
Epoch  90/100 - Loss (Binary Cross-Entropy): 0.316339
Epoch 100/100 - Loss (Binary Cross-Entropy): 0.316515

Evaluation Metrics on Test Set:
Test Accuracy: 84.80%
